In [2]:
from pathlib import Path
import shutil

import pandas as pd

In [3]:
DATA_DIR = Path("../data")

PROCESSED_DIR = DATA_DIR / "processed"
CONSUM_DIR = DATA_DIR / "consum"

CONSUM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [4]:
customers = pd.read_parquet(
    PROCESSED_DIR / "customers.parquet"
)

articles = pd.read_parquet(
    PROCESSED_DIR / "articles.parquet"
)

transactions = pd.read_parquet(
    PROCESSED_DIR / "transactions.parquet"
)

demo_users = pd.read_parquet(
    PROCESSED_DIR / "demo_users.parquet"
)

In [5]:
transactions = transactions[
    transactions["customer_id"].isin(
        demo_users["customer_id"]
    )
]

customers = customers[
    customers["customer_id"].isin(
        demo_users["customer_id"]
    )
]

In [6]:
user_data = transactions.merge(
    articles[
        [
            "article_id",
            "product_group_name",
            "colour_group_name",
            "product_type_name"
        ]
    ],
    on="article_id",
    how="left")

In [7]:
user_profiles = (
    user_data
    .groupby("customer_id")
    .agg(
        total_purchases=(
            "article_id",
            "count"
        ),
        avg_spend=(
            "price",
            "mean"
        ),
        favorite_category=(
            "product_group_name",
            lambda x: x.mode().iloc[0]
        ),
        favorite_color=(
            "colour_group_name",
            lambda x: x.mode().iloc[0]
        ),
        favorite_product_type=(
            "product_type_name",
            lambda x: x.mode().iloc[0]
        )
    )
    .reset_index()
)

In [8]:
user_profiles = user_profiles.merge(
    customers[
        [
            "customer_id",
            "age"
        ]
    ],
    on="customer_id",
    how="left"
)

In [9]:
user_profiles = user_profiles[
    [
        "customer_id",
        "age",
        "total_purchases",
        "avg_spend",
        "favorite_category",
        "favorite_color",
        "favorite_product_type"
    ]
]

In [10]:
user_profiles.to_parquet(
    CONSUM_DIR /
    "user_profiles.parquet",
    index=False
)

In [11]:
files_to_copy = [
    "articles.parquet",
    "customers.parquet",
    "demo_users.parquet",
    "optimized_recommendations.parquet",
    "bought_together.parquet"
]

In [12]:
for file_name in files_to_copy:

    shutil.copy2(
        PROCESSED_DIR / file_name,
        CONSUM_DIR / file_name
    )

In [13]:
for file in sorted(CONSUM_DIR.iterdir()):
    print(file.name)

articles.parquet
bought_together.parquet
customers.parquet
demo_users.parquet
optimized_recommendations.parquet
user_profiles.parquet
